# 06.01 - Neo4j Profile Example (with Sample Data)

This notebook demonstrates profiling functionality with automatic sample data population.

**Contents**:
1. Connect to Neo4j
2. Check APOC availability ⚠️
3. Populate sample data (Person, Movie, ACTED_IN)
4. Extract and display profile
5. Validate against a model

**Best for**: Learning, testing, understanding profiling workflows

## Connect to Database

In [ ]:
from utils import load_env, print_apoc_status, check_apoc_available
from neo4j import GraphDatabase

neo4j_uri = load_env("NEO4J_URI", "bolt://localhost:7687")
neo4j_user = load_env("NEO4J_USER", "neo4j")
neo4j_password = load_env("NEO4J_PASSWORD", "password")

print(f"Connecting to {neo4j_uri}...")
driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))
print("✓ Connected")

## Check APOC Status

⚠️ **Important**: APOC determines if property types are detected

In [ ]:
print_apoc_status(driver)

if not check_apoc_available(driver):
    print("\n⚠️  WARNING: APOC not installed on this Neo4j instance")
    print("   Property types will NOT be detected in profiles")
    print("   See PROFILE_TESTING_README.md for installation instructions")
    print("   You can still use all other profiling features")
else:
    print("\n✓ APOC is available - property types will be detected")

## Populate Sample Data

In [ ]:
from orthograph.cypher.generator import CypherGenerator
from orthograph.graph_definition.graph_definition import GraphDefinition
from orthograph.graph_definition.models import NodeModel, RelationshipModel
from typing import Optional

class Person(NodeModel):
    __label__ = "Person"
    __uid_field__ = "name"
    name: str
    born: Optional[int] = None

class Movie(NodeModel):
    __label__ = "Movie"
    __uid_field__ = "title"
    title: str
    year: int

class ActedIn(RelationshipModel):
    __label__ = "ACTED_IN"
    __source_label__ = "Person"
    __target_label__ = "Movie"
    role: str

graph_definition = GraphDefinition(
    name="Filmography",
    node_types=[Person, Movie],
    relationship_types=[ActedIn],
)

gen = CypherGenerator(graph_definition)

print("=== Creating Constraints ===")
for stmt in gen.generate_constraints():
    print(f"  {stmt}")
    driver.execute_query(stmt)

print("\n=== Creating Sample Nodes ===")
people = [
    {"__label__": "Person", "name": "Keanu Reeves", "born": 1964},
    {"__label__": "Person", "name": "Carrie-Anne Moss", "born": 1967},
    {"__label__": "Person", "name": "Lana Wachowski", "born": 1965},
]
movies = [
    {"__label__": "Movie", "title": "The Matrix", "year": 1999},
    {"__label__": "Movie", "title": "The Matrix Reloaded", "year": 2003},
]

for node_data in people + movies:
    query, params = gen.merge_node(node_data)
    driver.execute_query(query, **params)

print(f"✓ Created {len(people)} Person nodes")
print(f"✓ Created {len(movies)} Movie nodes")

print("\n=== Creating Relationships ===")
relationships = [
    {"__label__": "ACTED_IN", "__source_uid__": "Keanu Reeves", "__target_uid__": "The Matrix", "role": "Neo"},
    {"__label__": "ACTED_IN", "__source_uid__": "Carrie-Anne Moss", "__target_uid__": "The Matrix", "role": "Trinity"},
    {"__label__": "ACTED_IN", "__source_uid__": "Lana Wachowski", "__target_uid__": "The Matrix", "role": "The Oracle"},
]

for rel_data in relationships:
    query, params = gen.create_relationship(rel_data)
    driver.execute_query(query, **params)

print(f"✓ Created {len(relationships)} ACTED_IN relationships")
print("\n✓ Sample data populated successfully")

## Extract and Display Profile

In [ ]:
from utils import (
    extract_profile,
    display_profile_summary,
    display_node_profiles,
    display_relationship_profiles,
    display_constraints,
    export_profile_json,
)

profile = extract_profile(driver)
display_profile_summary(profile)
display_node_profiles(profile)
display_relationship_profiles(profile)
display_constraints(profile)

output_file = export_profile_json(profile)
print(f"\n✓ Profile exported to {output_file}")

## Validate Against Model

In [ ]:
from orthograph.api.database import validate

result = validate("neo4j", driver, graph_definition)

print("\n" + "=" * 80)
print("VALIDATION RESULT")
print("=" * 80)

print(f"Is Valid: {result.is_valid}")
print(f"Errors: {len(result.errors)}")
print(f"Warnings: {len(result.warnings)}")

if result.issues:
    print("\nIssues:")
    for issue in result.issues:
        severity_icon = "❌" if issue.severity.value == "ERROR" else "⚠️"
        print(f"  {severity_icon} [{issue.severity.value}] {issue.message}")
else:
    print("\n✓ Database matches the model perfectly!")

## Cleanup

In [ ]:
driver.close()
print("✓ Driver closed")